# Bundesliga Over/Under 2.5 Goals — EDA

**Focus:** Only pre-match information. Post-match stats are used as intermediate targets to understand *what drives goals*, not as model inputs.

1. Build the base table (matches + stats + lineups joined flat)
2. Team tiers (last 3 seasons standings)
3. Matchup type vs goals (Big/Mid/Small combinations)
4. Formations vs shots on target (raw, tier-controlled, same-team switching, matchups)
5. Form (last 5 games) vs goals
6. Summary — pre-match features correlation overview

In [ ]:
import sqlite3
import json
import re
import sys
import os
import warnings
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.getcwd())
from collector.normalize import normalize

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DB_PATH = 'data/raw.db'
SEASON_ORDER = ['2021/22', '2022/23', '2023/24', '2024/25', '2025/26']

---
## 1. Build the Base Table

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_matches   = pd.read_sql('SELECT * FROM matches',    conn)
df_stats     = pd.read_sql('SELECT * FROM statistics', conn)
df_lineups   = pd.read_sql('SELECT * FROM lineups',    conn)
df_standings = pd.read_sql('SELECT * FROM standings',  conn)
df_features  = pd.read_sql('SELECT * FROM features',   conn)
conn.close()

print(f'matches:    {len(df_matches):>5}')
print(f'statistics: {len(df_stats):>5}')
print(f'lineups:    {len(df_lineups):>5}')
print(f'standings:  {len(df_standings):>5}')

In [ ]:
# ── Parse matches ──────────────────────────────────────────────────────────
def parse_round(s):
    m = re.search(r'(\d+)$', str(s or ''))
    return int(m.group(1)) if m else None

match_rows = []
for _, r in df_matches.iterrows():
    d        = json.loads(r['raw_json'])
    fixture  = d.get('fixture', {})
    goals    = d.get('goals', {})
    date_str = fixture.get('date', '')
    match_rows.append({
        'match_id':      r['match_id'],
        'season':        r['season'],
        'match_date':    date_str[:10] if date_str else None,
        'round_number':  parse_round(d.get('league', {}).get('round', '')),
        'home_team':     r['home_team'],
        'away_team':     r['away_team'],
        'goals_home':    goals.get('home'),
        'goals_away':    goals.get('away'),
    })

base = pd.DataFrame(match_rows)
base['match_date']    = pd.to_datetime(base['match_date'])
base['total_goals']   = base['goals_home'] + base['goals_away']
base['target_over25'] = (base['total_goals'] > 2).astype(int)
print(f'Base: {base.shape}')
base.head(3)

In [ ]:
# ── Parse statistics (post-match — used as intermediate analysis targets, NOT model inputs) ──
STAT_NAMES = [
    'shots_on_target', 'shots_off_target', 'shots_total', 'shots_blocked',
    'shots_inside_box', 'shots_outside_box', 'fouls', 'corners', 'offsides',
    'possession', 'yellow_cards', 'red_cards', 'saves',
    'passes_total', 'passes_accurate', 'pass_pct',
]

def safe_num(val):
    if val is None: return None
    try: return float(str(val).replace('%', '').strip())
    except: return None

stat_rows = []
for _, r in df_stats.iterrows():
    d   = json.loads(r['raw_json'])
    rec = {'match_id': r['match_id']}
    for side_idx, prefix in [(0, 'h_'), (1, 'a_')]:
        try:    stats = d[side_idx]['statistics']
        except: stats = []
        for i, name in enumerate(STAT_NAMES):
            rec[f'{prefix}{name}'] = safe_num(stats[i]['value']) if i < len(stats) else None
    stat_rows.append(rec)

df_stats_flat = pd.DataFrame(stat_rows)

In [ ]:
# ── Parse lineups (PRE-MATCH) ──────────────────────────────────────────────
def parse_formation(f):
    if not f or not isinstance(f, str): return None, None, None
    parts = f.split('-')
    if len(parts) < 2: return None, None, None
    try:
        defenders   = int(parts[0])
        midfielders = sum(int(p) for p in parts[1:-1]) if len(parts) > 2 else int(parts[1])
        forwards    = int(parts[-1])
        return defenders, midfielders, forwards
    except: return None, None, None

lineup_rows = []
for _, r in df_lineups.iterrows():
    d = json.loads(r['raw_json'])
    try:    h_form, a_form = d[0].get('formation'), d[1].get('formation')
    except: h_form = a_form = None
    hd, hm, hf = parse_formation(h_form)
    ad, am, af = parse_formation(a_form)
    lineup_rows.append({
        'match_id': r['match_id'],
        'h_formation': h_form, 'a_formation': a_form,
        'h_defenders': hd, 'h_midfielders': hm, 'h_forwards': hf,
        'a_defenders': ad, 'a_midfielders': am, 'a_forwards': af,
    })

df_lineups_flat = pd.DataFrame(lineup_rows)

In [ ]:
# ── Join everything ────────────────────────────────────────────────────────
df = (
    base
    .merge(df_stats_flat,   on='match_id', how='left')
    .merge(df_lineups_flat, on='match_id', how='left')
)

# Pull rolling features from the Phase 2 features table
roll_cols = [c for c in df_features.columns if 'roll' in c] + ['match_id']
df = df.merge(df_features[roll_cols], on='match_id', how='left')

print(f'Joined: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nNull summary (cols with any nulls):')
null_pct = (df.isnull().mean() * 100).round(1)
print(null_pct[null_pct > 0].sort_values(ascending=False).to_string())

---
## 2. Team Tiers

Sum each team's league rank across their last 3 completed seasons.  
- **< 15 pts** → Big  
- **15–36 pts** → Mid  
- **> 36 pts** → Small  

Teams not in a Bundesliga season (newly promoted/relegated) get rank **19** for that season, making them automatically lean toward Small.

In [ ]:
# Parse standings into {season: {canonical_team: rank}}
standings_by_season = {}
for _, row in df_standings.iterrows():
    d = json.loads(row['raw_json'])
    table = d[0]['league']['standings'][0]
    standings_by_season[row['season']] = {
        normalize(entry['team']['name']): entry['rank']
        for entry in table
    }

print('Standings parsed for:', list(standings_by_season.keys()))

In [ ]:
def get_prior_seasons(match_season, n=3):
    """Return up to n completed seasons before match_season."""
    try:
        idx = SEASON_ORDER.index(match_season)
    except ValueError:
        return []
    start = max(0, idx - n)
    return SEASON_ORDER[start:idx]

def compute_tier_score(team, prior_seasons):
    """Sum of ranks across 3 slots. Missing season = rank 19 (promoted/relegated)."""
    total = 0
    for i in range(3):
        if i < len(prior_seasons):
            rank = standings_by_season.get(prior_seasons[i], {}).get(team, 19)
        else:
            rank = 19
        total += rank
    return total

def classify_tier(score):
    if score < 15:  return 'Big'
    if score > 36:  return 'Small'
    return 'Mid'

# Compute per-match tiers
h_tier_scores, a_tier_scores = [], []
h_tiers, a_tiers = [], []

for _, row in df.iterrows():
    prior = get_prior_seasons(row['season'], n=3)
    hs = compute_tier_score(row['home_team'], prior)
    as_ = compute_tier_score(row['away_team'], prior)
    h_tier_scores.append(hs)
    a_tier_scores.append(as_)
    h_tiers.append(classify_tier(hs))
    a_tiers.append(classify_tier(as_))

df['h_tier_score'] = h_tier_scores
df['a_tier_score'] = a_tier_scores
df['h_tier'] = h_tiers
df['a_tier'] = a_tiers

# Matchup type (sorted so Big vs Mid == Mid vs Big)
tier_order = {'Big': 0, 'Mid': 1, 'Small': 2}
def matchup_label(h, a):
    pair = sorted([h, a], key=lambda t: tier_order[t])
    return f'{pair[0]} vs {pair[1]}'

df['matchup_type'] = [matchup_label(h, a) for h, a in zip(df['h_tier'], df['a_tier'])]

print('Home tier distribution:')
print(df['h_tier'].value_counts().to_string())
print(f'\nMatchup types:')
print(df['matchup_type'].value_counts().sort_index().to_string())

In [ ]:
# Show tier assignments per team (using latest full 3-season window)
latest_prior = get_prior_seasons('2025/26', n=3)
all_teams = sorted(df['home_team'].unique())

tier_table = []
for team in all_teams:
    score = compute_tier_score(team, latest_prior)
    ranks = [standings_by_season.get(s, {}).get(team, 19) for s in latest_prior]
    tier_table.append({'Team': team, 'Ranks': ranks, 'Score': score, 'Tier': classify_tier(score)})

tier_df = pd.DataFrame(tier_table).sort_values('Score')
print(f'Team tiers (based on {latest_prior}):\n')
for _, r in tier_df.iterrows():
    print(f"  {r['Score']:>3}  [{r['Tier']:<5}]  {r['Team']:<30}  ranks: {r['Ranks']}")

---
## 3. Matchup Type vs Goals
Do Big vs Small games produce more goals than Mid vs Mid?

In [ ]:
matchup_stats = (
    df.groupby('matchup_type')
    .agg(
        avg_goals=('total_goals', 'mean'),
        over_rate=('target_over25', 'mean'),
        avg_home_goals=('goals_home', 'mean'),
        avg_away_goals=('goals_away', 'mean'),
        n=('match_id', 'count'),
    )
    .sort_values('avg_goals', ascending=False)
)
matchup_stats['over_rate_pct'] = (matchup_stats['over_rate'] * 100).round(1)
print(matchup_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg total goals by matchup
ms = matchup_stats.sort_values('avg_goals')
axes[0].barh(ms.index, ms['avg_goals'], color='steelblue', edgecolor='white')
axes[0].axvline(df['total_goals'].mean(), color='crimson', ls='--', lw=1.5,
                label=f'Overall avg {df["total_goals"].mean():.2f}')
for i, (idx, row) in enumerate(ms.iterrows()):
    axes[0].text(row['avg_goals'] + 0.02, i, f"n={row['n']:.0f}", va='center', fontsize=9)
axes[0].set_title('Avg Total Goals by Matchup Type')
axes[0].set_xlabel('Avg Total Goals')
axes[0].legend()

# Over 2.5 rate by matchup
ms2 = matchup_stats.sort_values('over_rate_pct')
axes[1].barh(ms2.index, ms2['over_rate_pct'], color='steelblue', edgecolor='white')
overall_rate = df['target_over25'].mean() * 100
axes[1].axvline(overall_rate, color='crimson', ls='--', lw=1.5,
                label=f'Overall {overall_rate:.1f}%')
axes[1].set_title('Over 2.5 Rate by Matchup Type')
axes[1].set_xlabel('Over 2.5 %')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Directional: does it matter who is home?
dir_stats = (
    df.groupby(['h_tier', 'a_tier'])
    .agg(
        avg_goals=('total_goals', 'mean'),
        over_rate=('target_over25', 'mean'),
        n=('match_id', 'count'),
    )
    .round(3)
)
print('Directional matchup stats (Home tier → Away tier):\n')
print(dir_stats.to_string())

# Pivot for heatmap
pivot = dir_stats['avg_goals'].unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Avg Total Goals — Home Tier (row) vs Away Tier (col)')
ax.set_ylabel('Home Tier')
ax.set_xlabel('Away Tier')
plt.tight_layout()
plt.show()

---
## 4. Formations vs Shots on Target

**Chain:** Formation (pre-match) → shots on target (in-match) → goals  
Does the formation itself drive shots, or is it just team quality?

In [ ]:
# 4a. Raw: avg shots on target by home formation
form_shots = (
    df.groupby('h_formation')
    .agg(avg_sot=('h_shots_on_target', 'mean'), avg_goals=('total_goals', 'mean'), n=('match_id', 'count'))
    .query('n >= 20')
    .sort_values('avg_sot', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(form_shots.index, form_shots['avg_sot'], color='steelblue', edgecolor='white')
ax.axvline(df['h_shots_on_target'].mean(), color='crimson', ls='--', lw=1.5,
           label=f'Overall avg {df["h_shots_on_target"].mean():.1f}')
for i, (idx, row) in enumerate(form_shots.iterrows()):
    ax.text(row['avg_sot'] + 0.05, i, f"n={row['n']}", va='center', fontsize=8)
ax.set_title('4a — Avg Shots on Target by Home Formation (≥20 matches, raw)')
ax.set_xlabel('Avg Shots on Target')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4b. Controlled by tier: within each tier, does formation still matter?
tier_form = (
    df.groupby(['h_tier', 'h_formation'])
    .agg(avg_sot=('h_shots_on_target', 'mean'), n=('match_id', 'count'))
    .reset_index()
    .query('n >= 10')
    .sort_values(['h_tier', 'avg_sot'], ascending=[True, False])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, tier in zip(axes, ['Big', 'Mid', 'Small']):
    sub = tier_form[tier_form['h_tier'] == tier].sort_values('avg_sot', ascending=True)
    if len(sub) == 0:
        ax.set_title(f'{tier} — not enough data (≥10 matches)')
        continue
    ax.barh(sub['h_formation'], sub['avg_sot'], color='steelblue', edgecolor='white')
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(row['avg_sot'] + 0.05, i, f"n={row['n']}", va='center', fontsize=8)
    ax.set_title(f'4b — {tier} Teams: SoT by Formation')
    ax.set_xlabel('Avg Shots on Target')

plt.tight_layout()
plt.show()

In [ ]:
# 4c. Same-team switching: when a team changes formation, do their shots change?
team_form_stats = (
    df.groupby(['home_team', 'h_formation'])
    .agg(
        avg_sot=('h_shots_on_target', 'mean'),
        avg_goals=('goals_home', 'mean'),
        n=('match_id', 'count'),
    )
    .reset_index()
    .query('n >= 5')
)

# Teams with multiple qualifying formations
multi_form_teams = (
    team_form_stats.groupby('home_team')
    .filter(lambda g: len(g) >= 2)
    .sort_values(['home_team', 'avg_sot'], ascending=[True, False])
)

print('4c — Same team, different formations (each ≥5 home matches):\n')
for team in multi_form_teams['home_team'].unique():
    sub = multi_form_teams[multi_form_teams['home_team'] == team]
    sot_range = sub['avg_sot'].max() - sub['avg_sot'].min()
    print(f'  {team} (SoT spread: {sot_range:.1f})')
    for _, r in sub.iterrows():
        print(f"    {r['h_formation']:<12}  SoT={r['avg_sot']:.1f}  Goals={r['avg_goals']:.1f}  n={r['n']}")
    print()

In [ ]:
# 4d. Formation matchup: parsed numbers
# Defenders, midfielders, forwards vs shots on target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

combos = [
    ('h_defenders',   'h_shots_on_target', 'Home Defenders → Home SoT'),
    ('h_midfielders', 'h_shots_on_target', 'Home Midfielders → Home SoT'),
    ('h_forwards',    'h_shots_on_target', 'Home Forwards → Home SoT'),
    ('a_defenders',   'h_shots_on_target', 'Opp Defenders → Home SoT'),
    ('a_midfielders', 'h_shots_on_target', 'Opp Midfielders → Home SoT'),
    ('a_forwards',    'h_shots_on_target', 'Opp Forwards → Home SoT'),
]

for ax, (x_col, y_col, title) in zip(axes.flat, combos):
    agg = df.groupby(x_col)[y_col].agg(['mean', 'count']).reset_index()
    agg = agg[agg['count'] >= 10]
    r = df[[x_col, y_col]].corr().iloc[0, 1]
    ax.bar(agg[x_col].astype(int), agg['mean'], color='steelblue', edgecolor='white')
    ax.set_title(f'{title} (r={r:.2f})', fontsize=10)
    ax.set_xlabel(x_col)
    ax.set_ylabel('Avg Home SoT')

plt.suptitle('4d — Formation Numbers vs Home Shots on Target', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Forward surplus (home forwards - away defenders) → shots and goals
df['fwd_vs_def'] = df['h_forwards'] - df['a_defenders']
df['def_vs_fwd'] = df['h_defenders'] - df['a_forwards']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

agg1 = df.groupby('fwd_vs_def').agg(
    avg_h_sot=('h_shots_on_target', 'mean'), n=('match_id', 'count')
).reset_index().query('n >= 10')
r1 = df[['fwd_vs_def', 'h_shots_on_target']].corr().iloc[0, 1]
axes[0].bar(agg1['fwd_vs_def'].astype(int), agg1['avg_h_sot'], color='steelblue', edgecolor='white')
axes[0].set_title(f'Home Fwd − Away Def → Home SoT (r={r1:.2f})')
axes[0].set_xlabel('Forward Surplus')
axes[0].set_ylabel('Avg Home SoT')

agg2 = df.groupby('fwd_vs_def').agg(
    avg_goals=('total_goals', 'mean'), n=('match_id', 'count')
).reset_index().query('n >= 10')
r2 = df[['fwd_vs_def', 'total_goals']].corr().iloc[0, 1]
axes[1].bar(agg2['fwd_vs_def'].astype(int), agg2['avg_goals'], color='tomato', edgecolor='white')
axes[1].set_title(f'Home Fwd − Away Def → Total Goals (r={r2:.2f})')
axes[1].set_xlabel('Forward Surplus')
axes[1].set_ylabel('Avg Total Goals')

plt.tight_layout()
plt.show()

---
## 5. Form (Last 5 Games) vs Goals

Rolling stats from the `features` table — computed with strict no-leakage (only matches before each row).

In [ ]:
# Filter to rows with form data
df_form = df[df['h_roll_games_available'] > 0].copy()
print(f'Rows with form data: {len(df_form)} / {len(df)}')

# Correlate rolling features with total goals
roll_features = [
    'h_roll_goals_scored', 'h_roll_goals_conceded', 'h_roll_total_goals',
    'h_roll_over25_rate', 'h_roll_shots_on_target', 'h_roll_shots_total',
    'h_roll_corners', 'h_roll_possession', 'h_roll_wins', 'h_roll_draws',
    'a_roll_goals_scored', 'a_roll_goals_conceded', 'a_roll_total_goals',
    'a_roll_over25_rate', 'a_roll_shots_on_target', 'a_roll_shots_total',
    'a_roll_corners', 'a_roll_possession', 'a_roll_wins', 'a_roll_draws',
]

corr_form = (
    df_form[roll_features + ['total_goals']]
    .corr()['total_goals']
    .drop('total_goals')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 8))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr_form.values]
ax.barh(corr_form.index, corr_form.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Rolling Form Features — Correlation with Total Goals')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

In [ ]:
# Do teams with high rolling SoT actually score more in the next game?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Home rolling SoT → home goals
bins1 = pd.qcut(df_form['h_roll_shots_on_target'].dropna(), 5, duplicates='drop')
df_form['h_sot_bin'] = bins1
agg = df_form.groupby('h_sot_bin', observed=True)['goals_home'].mean()
axes[0].bar(range(len(agg)), agg.values, color='steelblue', edgecolor='white')
axes[0].set_xticks(range(len(agg)))
axes[0].set_xticklabels([f'{i.left:.1f}–{i.right:.1f}' for i in agg.index], rotation=20, fontsize=8)
axes[0].set_title('Rolling Home SoT → Home Goals Scored')
axes[0].set_ylabel('Avg Home Goals')

# Home rolling win rate → total goals
agg2 = df_form.groupby(pd.cut(df_form['h_roll_wins'], bins=5))['total_goals'].mean()
axes[1].bar(range(len(agg2)), agg2.values, color='steelblue', edgecolor='white')
axes[1].set_xticks(range(len(agg2)))
axes[1].set_xticklabels([f'{i.left:.1f}–{i.right:.1f}' for i in agg2.index], rotation=20, fontsize=8)
axes[1].set_title('Rolling Home Win Rate → Total Goals')
axes[1].set_ylabel('Avg Total Goals')

# Combined rolling over25 rate → actual over25
df_form['combined_roll_over25'] = (
    df_form['h_roll_over25_rate'].fillna(0) + df_form['a_roll_over25_rate'].fillna(0)
) / 2
agg3 = df_form.groupby(
    pd.qcut(df_form['combined_roll_over25'], 5, duplicates='drop')
)['target_over25'].mean()
axes[2].bar(range(len(agg3)), agg3.values * 100, color='steelblue', edgecolor='white')
axes[2].set_xticks(range(len(agg3)))
axes[2].set_xticklabels([f'{i.left:.2f}–{i.right:.2f}' for i in agg3.index], rotation=20, fontsize=8)
axes[2].set_title('Combined Rolling Over2.5 Rate → Actual Over2.5 %')
axes[2].set_ylabel('Over 2.5 %')

plt.tight_layout()
plt.show()

In [ ]:
# Does form interact with tier?
df_form['h_form_class'] = pd.cut(
    df_form['h_roll_wins'], bins=[-0.01, 0.3, 0.6, 1.01],
    labels=['Poor', 'Average', 'Good']
)

form_tier = (
    df_form.groupby(['h_tier', 'h_form_class'], observed=True)
    .agg(
        avg_goals=('total_goals', 'mean'),
        over_rate=('target_over25', 'mean'),
        n=('match_id', 'count')
    )
    .round(3)
)
print('Tier × Form → Goals:\n')
print(form_tier.to_string())

# Heatmap
pivot_ft = form_tier['avg_goals'].unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot_ft, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Avg Total Goals — Tier (row) × Form (col)')
ax.set_ylabel('Home Tier')
ax.set_xlabel('Home Form (rolling win rate)')
plt.tight_layout()
plt.show()

---
## 6. Summary — Pre-Match Feature Correlations

All features that are genuinely available before kick-off, correlated against both targets.

In [ ]:
pre_match_features = [
    # Tier & matchup
    'h_tier_score', 'a_tier_score',
    # Formation (parsed numbers)
    'h_defenders', 'h_midfielders', 'h_forwards',
    'a_defenders', 'a_midfielders', 'a_forwards',
    'fwd_vs_def', 'def_vs_fwd',
    # Rolling form (last 5 games)
    'h_roll_goals_scored', 'h_roll_goals_conceded', 'h_roll_total_goals',
    'h_roll_over25_rate', 'h_roll_shots_on_target', 'h_roll_possession',
    'h_roll_wins',
    'a_roll_goals_scored', 'a_roll_goals_conceded', 'a_roll_total_goals',
    'a_roll_over25_rate', 'a_roll_shots_on_target', 'a_roll_possession',
    'a_roll_wins',
]

corr_all = (
    df[pre_match_features + ['total_goals', 'target_over25']]
    .corr()[['total_goals', 'target_over25']]
    .drop(['total_goals', 'target_over25'])
    .rename(columns={'total_goals': 'r_total_goals', 'target_over25': 'r_over25'})
    .sort_values('r_over25', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 9))
y = range(len(corr_all))
ax.barh(y, corr_all['r_over25'], height=0.4, label='r with over2.5', color='steelblue', alpha=0.8)
ax.barh([i + 0.4 for i in y], corr_all['r_total_goals'], height=0.4, label='r with total goals', color='tomato', alpha=0.8)
ax.set_yticks([i + 0.2 for i in y])
ax.set_yticklabels(corr_all.index, fontsize=8)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Pre-Match Features — Correlation with Targets')
ax.set_xlabel('Pearson r')
ax.legend()
plt.tight_layout()
plt.show()

print(corr_all.round(3).to_string())